# `catalog.jsonl` 商品数据与公开标签探索

这份 Notebook 用可逐格运行的方式查看商品目录的真实构成，并在最后把它与 `public_set.jsonl` 中的公开标签关联起来。

需要先区分两类数据：

- `catalog.jsonl`：50,000 条商品记录，本身不包含会话的正确答案标签；
- `public_set.jsonl`：200 个带标签的开发会话，包含 `ground_truth.parent_asin`、场景和难度等字段。

## 0. 运行准备

在 VS Code 中选择项目的 `.venv\Scripts\python.exe` 作为 Notebook 内核。如果提示缺少内核，可在激活 `.venv` 后运行：

```powershell
python -m pip install ipykernel
```

探索代码只使用标准库和项目现有代码，不依赖 pandas。

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint
import json
import random
import sys

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'data' / 'catalog.jsonl').exists():
            return candidate
    raise RuntimeError('找不到 data/catalog.jsonl，请从项目目录或 notebooks 目录运行。')

PROJECT_ROOT = find_project_root()
CATALOG_PATH = PROJECT_ROOT / 'data' / 'catalog.jsonl'
PUBLIC_SET_PATH = PROJECT_ROOT / 'data' / 'public_set.jsonl'
sys.path.insert(0, str(PROJECT_ROOT))

print('项目根目录:', PROJECT_ROOT)
print('Catalog:', CATALOG_PATH, f'({CATALOG_PATH.stat().st_size:,} bytes)')
print('Public set:', PUBLIC_SET_PATH)

## 1. 先看 JSONL 原始形态

JSONL 表示每一行都是一个独立 JSON 对象。先直接读取前两行，既看原始文本，也看解析后的字段名。

In [ ]:
with CATALOG_PATH.open(encoding='utf-8') as handle:
    first_two_lines = [next(handle).rstrip('\n') for _ in range(2)]

for index, raw_line in enumerate(first_two_lines, start=1):
    product = json.loads(raw_line)
    print(f'--- 第 {index} 行原始长度: {len(raw_line):,} 字符 ---')
    print(raw_line[:500] + ('...' if len(raw_line) > 500 else ''))
    print('字段:', list(product))

## 2. 加载全部 50,000 条商品

文件约 60 MB，全部加载便于反复探索。后续单元共用 `products` 和 `by_asin`，无需重复读取。

In [ ]:
with CATALOG_PATH.open(encoding='utf-8') as handle:
    products = [json.loads(line) for line in handle if line.strip()]

by_asin = {str(product['parent_asin']): product for product in products}
print('商品记录数:', len(products))
print('唯一 parent_asin:', len(by_asin))
assert len(products) == 50_000
assert len(products) == len(by_asin)

## 3. 查看一条完整商品记录

修改 `ROW_INDEX` 可以浏览任意位置。每条商品通常包含标题、特征、描述、价格、类别层级、详情、评分和店铺。

In [ ]:
ROW_INDEX = 0
pprint(products[ROW_INDEX], sort_dicts=False)

## 4. 字段、数据类型和覆盖率

这里统计所有记录实际出现过的字段、类型和非空比例。`null`、空字符串、空列表和空字典都按空值处理。

In [ ]:
all_fields = sorted({key for product in products for key in product})
empty_values = (None, '', [], {})
field_report = []

for field in all_fields:
    present = sum(field in product for product in products)
    nonempty = sum(product.get(field) not in empty_values for product in products)
    types = Counter(type(product.get(field)).__name__ for product in products if field in product)
    field_report.append({
        'field': field,
        'present': present,
        'nonempty': nonempty,
        'nonempty_pct': round(100 * nonempty / len(products), 2),
        'types': dict(types),
    })

for row in field_report:
    print(f"{row['field']:<16} present={row['present']:>5}  nonempty={row['nonempty']:>5} "
          f"({row['nonempty_pct']:>6.2f}%)  types={row['types']}")

## 5. 随机抽样，不只看文件开头

固定随机种子，保证每次运行得到相同样本。这里只显示紧凑摘要；需要完整内容时可用上一格的 `pprint`。

In [ ]:
rng = random.Random(2026)
sampled_products = rng.sample(products, 5)
for product in sampled_products:
    pprint({
        'parent_asin': product.get('parent_asin'),
        'title': product.get('title'),
        'price': product.get('price'),
        'categories': product.get('categories'),
        'average_rating': product.get('average_rating'),
        'rating_number': product.get('rating_number'),
        'store': product.get('store'),
    })
    print()

## 6. 类别标签的层级构成

`categories` 是从大类到细类的列表。这里分别统计完整类别路径、最末级类别和层级深度。

In [ ]:
category_paths = Counter()
leaf_categories = Counter()
category_depths = Counter()

for product in products:
    values = [str(value) for value in (product.get('categories') or [])]
    if values:
        category_paths[' > '.join(values)] += 1
        leaf_categories[values[-1]] += 1
        category_depths[len(values)] += 1

print('最常见的 15 个末级类别:')
pprint(leaf_categories.most_common(15))
print('\n类别层级深度分布:')
pprint(sorted(category_depths.items()))
print('\n最常见的 10 条完整路径:')
pprint(category_paths.most_common(10))

## 7. 价格、评分和评论数

下面只对非空数值计算分位数。极端值也会显示，便于判断后续是否需要截断或对数变换。

In [ ]:
def numeric_summary(values):
    clean = []
    invalid = Counter()
    for value in values:
        if value is None:
            continue
        try:
            clean.append(float(value))
        except (TypeError, ValueError):
            invalid[str(value)] += 1
    clean.sort()
    if not clean:
        return {}
    def percentile(p):
        index = round((len(clean) - 1) * p)
        return clean[index]
    return {
        'numeric_count': len(clean),
        'invalid_count': sum(invalid.values()),
        'invalid_examples': invalid.most_common(5),
        'min': clean[0],
        'p25': percentile(0.25),
        'median': percentile(0.50),
        'p75': percentile(0.75),
        'p95': percentile(0.95),
        'max': clean[-1],
    }

print('price:')
pprint(numeric_summary(product.get('price') for product in products))
print('average_rating:')
pprint(numeric_summary(product.get('average_rating') for product in products))
print('rating_number:')
pprint(numeric_summary(product.get('rating_number') for product in products))

## 8. `features` 和 `details` 的构成

`features` 通常是自由文本列表，`details` 是键值字典。Details 的键并不完全统一，因此先统计最常见的键，再查看真实样本。

In [ ]:
detail_keys = Counter()
feature_lengths = Counter()
for product in products:
    details = product.get('details') or {}
    if isinstance(details, dict):
        detail_keys.update(details.keys())
    features = product.get('features') or []
    feature_lengths[len(features) if isinstance(features, list) else 0] += 1

print('最常见的 20 个 details 键:')
pprint(detail_keys.most_common(20))
print('\nfeatures 数量分布（前 15 个长度）:')
pprint(sorted(feature_lengths.items())[:15])

example = next(product for product in products if product.get('features') and product.get('details'))
print('\n一条真实样本:')
pprint({'parent_asin': example['parent_asin'], 'features': example['features'], 'details': example['details']})

## 9. 按 ASIN 精确查看商品

把 `ASIN_TO_INSPECT` 改成任意 `parent_asin`。如果想查看公开会话的正确答案商品，可以在最后一节复制目标 ASIN 到这里。

In [ ]:
ASIN_TO_INSPECT = products[0]['parent_asin']
selected = by_asin.get(ASIN_TO_INSPECT)
if selected is None:
    print('目录中不存在:', ASIN_TO_INSPECT)
else:
    pprint(selected, sort_dicts=False)

## 10. 按关键词搜索商品内容

这是一个用于人工检查的简单包含匹配，不是 BM25。修改 `QUERY`，可以搜索标题、特征、描述、类别、详情和店铺。多个词必须全部出现。

In [ ]:
def flatten_text(value):
    if value is None:
        return ''
    if isinstance(value, dict):
        return ' '.join(f'{key} {item}' for key, item in value.items())
    if isinstance(value, list):
        return ' '.join(map(str, value))
    return str(value)

def search_catalog(query, limit=10):
    terms = [term.casefold() for term in query.split() if term.strip()]
    results = []
    fields = ('title', 'features', 'description', 'categories', 'details', 'store')
    for product in products:
        haystack = ' '.join(flatten_text(product.get(field)) for field in fields).casefold()
        if all(term in haystack for term in terms):
            results.append(product)
            if len(results) >= limit:
                break
    return results

QUERY = 'leather belt'
matches = search_catalog(QUERY, limit=10)
print(f'查询 {QUERY!r}，展示 {len(matches)} 条:')
for product in matches:
    pprint({
        'parent_asin': product['parent_asin'],
        'title': product.get('title'),
        'price': product.get('price'),
        'categories': product.get('categories'),
    })

## 11. 从商品记录生成 evaluator 的 `intent_card`

这一步能直观看到 evaluator 如何把商品属性转成隐藏硬约束和软偏好。注意：这是本地理解实验，正式 Agent 不会直接得到目标商品或完整 intent card。

In [ ]:
from evaluator.local_evaluator import intent_card

for product in sampled_products[:3]:
    print('---', product['parent_asin'], product.get('title'), '---')
    pprint(intent_card(product))

## 12. 查看真正的公开会话标签，并关联目标商品

`public_set.jsonl` 才包含用于本地评估的标签：

- `ground_truth.parent_asin`：正确目标；
- `scenario_type`：Buying、Browsing、Intent Override 或 Boundary；
- `category_bucket`：类别分桶；
- `difficulty_bucket`：难度分桶；
- `user_profile`：匿名聚合画像。

下面读取这些标签，并用目标 ASIN 回查 Catalog。

In [ ]:
with PUBLIC_SET_PATH.open(encoding='utf-8') as handle:
    sessions = [json.loads(line) for line in handle if line.strip()]

print('公开标签会话数:', len(sessions))
print('场景分布:', Counter(item['scenario_type'] for item in sessions))
print('类别分桶:', Counter(item.get('category_bucket') for item in sessions))
print('难度分桶:', Counter(item.get('difficulty_bucket') for item in sessions))

LABEL_INDEX = 0
labeled_session = sessions[LABEL_INDEX]
target_asin = str(labeled_session['ground_truth']['parent_asin'])
print('\n会话标签:')
pprint(labeled_session)
print('\n标签指向的 Catalog 商品:')
pprint(by_asin[target_asin], sort_dicts=False)

## 13. 后续可以亲自修改的观察点

1. 修改 `ROW_INDEX`，比较不同商品字段是否稳定。
2. 修改 `QUERY`，观察普通包含匹配容易产生哪些噪声。
3. 修改 `ASIN_TO_INSPECT`，查看特定商品的完整属性。
4. 修改 `LABEL_INDEX`，比较不同场景和难度的目标商品。
5. 检查 `intent_card(product)` 是否会截断、误分类或遗漏重要属性。
6. 比较 Catalog 的类别路径与 `category_bucket`，理解细粒度商品类别如何被分桶。

不要在正式 Agent 中直接读取 `public_set.jsonl` 的 `ground_truth`；该字段只用于本地开发评分，私有评估不会把正确答案传给 Agent。